<a href="https://colab.research.google.com/github/Dilukshika-Sasitharan/Statistical-Learning-e23355/blob/main/E23355_Assignment_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment : 6**
# **E/23/355**
---


## Q. Analytical Derivation

**Model**
$$x_k^- = A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1}, \qquad y_k^- = H_k x_k^- + z_k$$
$$x_{k-1}^+\sim N(m_{k-1},P_{k-1}),\quad w_{k-1}\sim N(0,\Sigma_p),\quad z_k\sim N(0,\Sigma_m)$$
all mutually independent.



### 1. Prediction distribution of $x_k^-$
$x_k^-$ is a linear function of the jointly independent Gaussian vectors $x_{k-1}^+$ and $w_{k-1}$, hence Gaussian. Taking expectations and using independence of the two terms when computing the covariance of the sum:
$$m_k^- = \mathbb{E}[x_k^-]=A_{k-1}m_{k-1}+G_{k-1}\cdot 0 = A_{k-1}m_{k-1}$$
$$P_k^- = \mathrm{Cov}(x_k^-) = A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_pG_{k-1}^T$$
(cross terms vanish because $x_{k-1}^+\perp w_{k-1}$). So $x_k^-\sim N(m_k^-,P_k^-)$.



### 2. Predictive measurement distribution
$y_k^-=H_kx_k^-+z_k$ is linear in independent Gaussians $x_k^-$ and $z_k$, hence Gaussian:
$$y_k^-\sim N(H_km_k^-,\; H_kP_k^-H_k^T+\Sigma_m)$$



### 3. Joint distribution of $(x_k^-,y_k^-)$
Write $\begin{bmatrix}x_k^-\\y_k^-\end{bmatrix}=\begin{bmatrix}I&0\\H_k&I\end{bmatrix}\begin{bmatrix}x_k^-\\z_k\end{bmatrix}$, a linear map of independent Gaussians, hence jointly Gaussian. Computing blocks:
- $\mathrm{Var}(x_k^-)=P_k^-$
- $\mathrm{Cov}(x_k^-,y_k^-)=\mathrm{Cov}(x_k^-,H_kx_k^-+z_k)=P_k^-H_k^T$
- $\mathrm{Var}(y_k^-)=H_kP_k^-H_k^T+\Sigma_m$

which reproduces exactly the stated joint covariance matrix.



### 4. Update step
For a joint Gaussian $\begin{bmatrix}x\\y\end{bmatrix}\sim N\left(\begin{bmatrix}m_x\\m_y\end{bmatrix},\begin{bmatrix}P_{xx}&P_{xy}\\P_{yx}&P_{yy}\end{bmatrix}\right)$, the conditional law $x\mid y=y_{obs}$ is Gaussian with mean $m_x+P_{xy}P_{yy}^{-1}(y_{obs}-m_y)$ and covariance $P_{xx}-P_{xy}P_{yy}^{-1}P_{yx}$ (obtained by completing the square in $x$ in the joint density, using the Schur-complement form of the inverse of a block covariance matrix). Substituting the blocks from Step 3:
$$K_k\triangleq P_k^-H_k^T(H_kP_k^-H_k^T+\Sigma_m)^{-1}$$
$$m_k = m_k^- + K_k(y_k^{obs}-H_km_k^-),\qquad P_k=(I-K_kH_k)P_k^-$$
so $x_k^+\triangleq(x_k^-\mid y_k^-=y_k^{obs})\sim N(m_k,P_k)$.



### 5.
$$\mathbb{E}[x_k^-\mid y_k^-=y_k^{obs}]=m_k,\qquad \mathrm{Var}(x_k^-\mid y_k^-=y_k^{obs})=P_k$$

## Q. 1-D Example

Scalar specialization with $A=a$, $G=1$, $\Sigma_p=q$, $H=h$, $\Sigma_m=r$.

**1.** $m_k^- = a\,m_{k-1}$, $\quad P_k^- = a^2P_{k-1}+q$.

**2.** With $S_k \triangleq h^2P_k^- + r$ and $K_k = P_k^- h / S_k$, and $v_k\triangleq y_k^{obs}-hm_k^-$:
$$m_k = m_k^- + K_kv_k = m_k^- + \frac{P_k^-h}{S_k}(y_k^{obs}-hm_k^-), \qquad P_k=(1-K_kh)P_k^-=\Big(1-\frac{P_k^-h^2}{S_k}\Big)P_k^-$$

**3.** Predictive measurement distribution: $p(y_k^-\mid Y_{k-1}) = N(h\,m_k^-,\ h^2P_k^-+r)$.

**4.** Posterior-predictive measurement distribution: $p(y_k\mid Y_k) = N(h\,m_k,\ h^2P_k+r)$.

**5.** Numerical / animated demonstration in the code cells below.

In [ ]:
# 1-D Kalman filter: numerical example + animation
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# ---- numerical values ----
a, q, h, r = 0.95, 0.05, 1.0, 0.3
m0, P0 = 0.0, 1.0
n_steps = 30
rng = np.random.default_rng(42)

# simulate ground-truth state and measurements
x_true = np.zeros(n_steps)
x_true[0] = m0 + rng.normal(0, np.sqrt(P0))
for k in range(1, n_steps):
    x_true[k] = a * x_true[k-1] + rng.normal(0, np.sqrt(q))
y_obs = h * x_true + rng.normal(0, np.sqrt(r), size=n_steps)

# run the scalar Kalman filter, storing prior/posterior at every step
m_prior, P_prior = np.zeros(n_steps), np.zeros(n_steps)
m_post, P_post = np.zeros(n_steps), np.zeros(n_steps)
m_prev, P_prev = m0, P0
for k in range(n_steps):
    # predict
    mk_minus = a * m_prev
    Pk_minus = a**2 * P_prev + q
    # update
    Sk = h**2 * Pk_minus + r
    Kk = Pk_minus * h / Sk
    mk = mk_minus + Kk * (y_obs[k] - h * mk_minus)
    Pk = (1 - Kk * h) * Pk_minus
    m_prior[k], P_prior[k] = mk_minus, Pk_minus
    m_post[k], P_post[k] = mk, Pk
    m_prev, P_prev = mk, Pk

print('Final estimate: mean =', m_post[-1], ' var =', P_post[-1])

Final estimate: mean = 0.3080446719234481  var = 0.09228603058963326


In [ ]:
# Animated figure: prior (before update) vs posterior (after update) density at each step
x_grid = np.linspace(-6, 6, 400)
frames = []
for k in range(n_steps):
    prior_pdf = norm.pdf(x_grid, m_prior[k], np.sqrt(P_prior[k]))
    post_pdf  = norm.pdf(x_grid, m_post[k],  np.sqrt(P_post[k]))
    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_grid, y=prior_pdf, mode='lines', name='Prior  N(m_k^-, P_k^-)', line=dict(color='royalblue')),
            go.Scatter(x=x_grid, y=post_pdf,  mode='lines', name='Posterior  N(m_k, P_k)', line=dict(color='firebrick')),
            go.Scatter(x=[x_true[k]], y=[0], mode='markers', name='True state', marker=dict(color='black', size=10, symbol='x')),
        ],
        name=str(k)
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title='1-D Kalman Filter: Prior vs Posterior Density Evolution',
        xaxis=dict(title='x', range=[-6, 6]),
        yaxis=dict(title='density', range=[0, 2]),
        updatemenus=[dict(type='buttons', showactive=False,
                           buttons=[dict(label='Play', method='animate',
                                         args=[None, dict(frame=dict(duration=400, redraw=True), fromcurrent=True)]),
                                    dict(label='Pause', method='animate',
                                         args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])])],
        sliders=[dict(steps=[dict(method='animate', args=[[str(k)], dict(mode='immediate', frame=dict(duration=0, redraw=True))], label=str(k)) for k in range(n_steps)])]
    )
)
fig.show()

## Q. 2D Position Estimation — Part A

State $x_k=[p_x,p_y,v_x,v_y]^T$. Using $p(k)=p(k-1)+\Delta t\,v(k-1)+\tfrac12\Delta t^2 w$ and $v(k)=v(k-1)+\Delta t\,w$ in each coordinate (the constant-acceleration / impulse noise model) gives, in stacked form,
$$x_k^- = Ax_{k-1}^+ + Gw_{k-1}, \qquad A=\begin{bmatrix}1&0&\Delta t&0\\0&1&0&\Delta t\\0&0&1&0\\0&0&0&1\end{bmatrix},\quad G=\begin{bmatrix}\tfrac12\Delta t^2&0\\0&\tfrac12\Delta t^2\\\Delta t&0\\0&\Delta t\end{bmatrix}$$
Since the GPS only observes position,
$$y_k = Hx_k^- + z_k, \qquad H=\begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix}$$

## Q. 2D Position Estimation — Part B: Python Kalman Filter for GPS Tracking

In [ ]:
import numpy as np
import plotly.graph_objects as go

class KalmanFilter2D:
    """Constant-velocity Kalman filter for 2-D GPS position tracking."""
    def __init__(self, dt, sigma_p, sigma_m, m0, P0):
        self.dt = dt
        self.A = np.array([[1, 0, dt, 0],
                            [0, 1, 0, dt],
                            [0, 0, 1, 0],
                            [0, 0, 0, 1]])
        self.G = np.array([[0.5*dt**2, 0],
                            [0, 0.5*dt**2],
                            [dt, 0],
                            [0, dt]])
        self.H = np.array([[1, 0, 0, 0],
                            [0, 1, 0, 0]])
        self.Sigma_p = sigma_p * np.eye(2)   # process noise covariance (acceleration)
        self.Sigma_m = sigma_m * np.eye(2)   # measurement (GPS) noise covariance
        self.m = np.array(m0, dtype=float)
        self.P = np.array(P0, dtype=float)

    def predict(self):
        self.m = self.A @ self.m
        self.P = self.A @ self.P @ self.A.T + self.G @ self.Sigma_p @ self.G.T
        return self.m.copy(), self.P.copy()

    def update(self, y_obs):
        S = self.H @ self.P @ self.H.T + self.Sigma_m
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.m = self.m + K @ (y_obs - self.H @ self.m)
        self.P = (np.eye(4) - K @ self.H) @ self.P
        return self.m.copy(), self.P.copy()

    def step(self, y_obs):
        self.predict()
        return self.update(y_obs)

In [ ]:
# Simulate a true trajectory and noisy GPS measurements, then filter them
rng = np.random.default_rng(0)
dt = 1.0
n_steps = 60

A_true = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]])
G_true = np.array([[0.5*dt**2, 0], [0, 0.5*dt**2], [dt, 0], [0, dt]])
H_true = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
q_var, r_var = 0.01, 4.0   # process noise variance (m/s^2)^2 scale, GPS noise variance (m^2)

x_true = np.zeros((n_steps, 4))
x_true[0] = [0, 0, 1.0, 0.5]   # start at origin, velocity (1.0, 0.5) m/step
for k in range(1, n_steps):
    w = rng.normal(0, np.sqrt(q_var), size=2)
    x_true[k] = A_true @ x_true[k-1] + G_true @ w

y_meas = (H_true @ x_true.T).T + rng.normal(0, np.sqrt(r_var), size=(n_steps, 2))

# run the Kalman filter on the GPS measurements
kf = KalmanFilter2D(dt=dt, sigma_p=q_var, sigma_m=r_var,
                     m0=[0, 0, 0, 0], P0=np.eye(4)*10)
estimates = np.zeros((n_steps, 4))
covariances = np.zeros((n_steps, 4, 4))
for k in range(n_steps):
    m_k, P_k = kf.step(y_meas[k])
    estimates[k] = m_k
    covariances[k] = P_k

rmse = np.sqrt(np.mean(np.sum((estimates[:, :2] - x_true[:, :2])**2, axis=1)))
print('Position RMSE (filtered):', rmse)
print('Position RMSE (raw GPS):', np.sqrt(np.mean(np.sum((y_meas - x_true[:, :2])**2, axis=1))))

Position RMSE (filtered): 1.696927546942466
Position RMSE (raw GPS): 2.891565845675098


In [ ]:
# Plotly visualization: true path, noisy GPS, and filtered estimate
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines', name='True trajectory',
                          line=dict(color='black', width=3)))
fig.add_trace(go.Scatter(x=y_meas[:, 0], y=y_meas[:, 1], mode='markers', name='Noisy GPS measurements',
                          marker=dict(color='firebrick', size=6, opacity=0.6)))
fig.add_trace(go.Scatter(x=estimates[:, 0], y=estimates[:, 1], mode='lines+markers', name='Kalman filter estimate',
                          line=dict(color='royalblue', width=2), marker=dict(size=4)))
fig.update_layout(title='2-D Constant-Velocity GPS Tracking via Kalman Filter',
                   xaxis_title='x position (m)', yaxis_title='y position (m)',
                   yaxis=dict(scaleanchor='x', scaleratio=1))
fig.show()

In [ ]:
# Animated version: build up the trajectory step by step, showing the 1-sigma uncertainty ellipse
def cov_ellipse(mean, cov2x2, n_pts=60, n_std=1.0):
    vals, vecs = np.linalg.eigh(cov2x2)
    vals = np.maximum(vals, 0)
    theta = np.linspace(0, 2*np.pi, n_pts)
    circle = np.stack([np.cos(theta), np.sin(theta)])
    ellipse = vecs @ np.diag(n_std * np.sqrt(vals)) @ circle
    return mean[0] + ellipse[0], mean[1] + ellipse[1]

frames = []
for k in range(1, n_steps + 1):
    ex, ey = cov_ellipse(estimates[k-1, :2], covariances[k-1, :2, :2])
    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_true[:k, 0], y=x_true[:k, 1], mode='lines', name='True trajectory', line=dict(color='black', width=3)),
            go.Scatter(x=y_meas[:k, 0], y=y_meas[:k, 1], mode='markers', name='Noisy GPS', marker=dict(color='firebrick', size=6, opacity=0.6)),
            go.Scatter(x=estimates[:k, 0], y=estimates[:k, 1], mode='lines', name='Filtered estimate', line=dict(color='royalblue', width=2)),
            go.Scatter(x=ex, y=ey, mode='lines', name='1-sigma uncertainty', line=dict(color='royalblue', dash='dot')),
        ],
        name=str(k)
    ))

fig2 = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title='Animated Kalman Filter GPS Tracking',
        xaxis=dict(title='x position (m)', range=[x_true[:,0].min()-10, x_true[:,0].max()+10]),
        yaxis=dict(title='y position (m)', range=[x_true[:,1].min()-10, x_true[:,1].max()+10], scaleanchor='x', scaleratio=1),
        updatemenus=[dict(type='buttons', showactive=False,
                           buttons=[dict(label='Play', method='animate',
                                         args=[None, dict(frame=dict(duration=200, redraw=True), fromcurrent=True)]),
                                    dict(label='Pause', method='animate',
                                         args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])])],
    )
)
fig2.show()